# Session 12 Solutions: Designing Better Classes

Data Science with Python, Level 1, Stage 1: Python Foundations

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## In-Class Exercises

### In-Class Exercise 1: A Printable Point

In [1]:
import math

class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def distance_to(self, other):
        return math.hypot(self.x - other.x, self.y - other.y)

p, q = Point(0, 0), Point(3, 4)
print(p, q, p == Point(0, 0), p.distance_to(q))

Point(0, 0) Point(3, 4) True 5.0


### In-Class Exercise 2: Sortable Students

In [2]:
class Student:
    def __init__(self, name, gpa):
        self.name, self.gpa = name, gpa

    def __str__(self):
        return f"{self.name} ({self.gpa})"

    def __lt__(self, other):
        return self.gpa < other.gpa

group = [Student("Lina", 3.6), Student("Omar", 3.1), Student("Sara", 3.9), Student("Hadi", 2.8)]
for s in sorted(group, reverse=True):
    print(s)

Sara (3.9)
Lina (3.6)
Omar (3.1)
Hadi (2.8)


### In-Class Exercise 3: A Safe Thermostat

In [3]:
class Thermostat:
    def __init__(self, target=20):
        self.target = target

    @property
    def target(self):
        return self._target

    @target.setter
    def target(self, value):
        if not 10 <= value <= 30:
            raise ValueError(f"target must be between 10 and 30, got {value}")
        self._target = value

t = Thermostat()
t.target = 22
print(t.target)
try:
    t.target = 45
except ValueError as e:
    print("Error:", e)

22
Error: target must be between 10 and 30, got 45


### In-Class Exercise 4: Count the Objects

In [4]:
class Ticket:
    created = 0

    def __init__(self, holder):
        Ticket.created += 1
        self.number = Ticket.created
        self.holder = holder

tickets = [Ticket("Lina"), Ticket("Omar"), Ticket("Sara")]
for t in tickets:
    print(t.number, t.holder)
print("Total:", Ticket.created)

1 Lina
2 Omar
3 Sara
Total: 3


## Part A: Basic

### Exercise 1: A Money Class

In [5]:
class Money:
    def __init__(self, amount, currency):
        self.amount, self.currency = amount, currency

    def __str__(self):
        return f"{self.amount:.2f} {self.currency}"

    def __repr__(self):
        return f"Money({self.amount}, {self.currency!r})"

    def __eq__(self, other):
        return self.amount == other.amount and self.currency == other.currency

a, b, c = Money(45, "USD"), Money(45, "USD"), Money(45, "EUR")
print(a, repr(a))
print(a == b, a == c)

45.00 USD Money(45, 'USD')
True False


### Exercise 2: A Playlist

In [6]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []

    def add(self, song):
        self.songs.append(song)

    def __len__(self):
        return len(self.songs)

    def __str__(self):
        return f"{self.name}: {len(self)} songs"

p = Playlist("Study")
for s in ["Song A", "Song B", "Song C"]:
    p.add(s)
print(p, len(p))

Study: 3 songs 3


## Part B: Intermediate

### Exercise 3: Product with Validation

In [7]:
class Product:
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = value

    @property
    def quantity(self):
        return self._quantity

    @quantity.setter
    def quantity(self, value):
        if value < 0:
            raise ValueError("quantity cannot be negative")
        self._quantity = value

    def total(self):
        return self.price * self.quantity

print(Product("Notebook", 3.5, 4).total())
try:
    Product("Pen", -1, 5)
except ValueError as e:
    print("Error:", e)

14.0
Error: price cannot be negative


### Exercise 4: Employees from Text

In [8]:
class Employee:
    def __init__(self, name, department, salary):
        self.name, self.department, self.salary = name, department, salary

    @classmethod
    def from_text(cls, line):
        name, dept, salary = [p.strip() for p in line.split("|")]
        return cls(name, dept, int(salary))

    def __repr__(self):
        return f"Employee({self.name!r}, {self.department!r}, {self.salary})"

lines = ["Lina Haddad | Data | 2400", "Omar Saleh | Sales | 1800", "Sara Nassar | Engineering | 2900"]
staff = [Employee.from_text(l) for l in lines]
for e in sorted(staff, key=lambda e: e.salary):
    print(e)

Employee('Omar Saleh', 'Sales', 1800)
Employee('Lina Haddad', 'Data', 2400)
Employee('Sara Nassar', 'Engineering', 2900)


## Part C: Challenge

### Exercise 5: A Fraction Class

`__repr__ = __str__` reuses the same method for both, so lists of fractions print nicely too.

In [9]:
import math

class Fraction:
    def __init__(self, num, den):
        if den == 0:
            raise ValueError("denominator cannot be zero")
        g = math.gcd(num, den)
        self.num, self.den = num // g, den // g

    def __str__(self):
        return f"{self.num}/{self.den}"

    __repr__ = __str__

    def __eq__(self, other):
        return self.num == other.num and self.den == other.den

    def __lt__(self, other):
        return self.num * other.den < other.num * self.den

    def __add__(self, other):
        return Fraction(self.num * other.den + other.num * self.den, self.den * other.den)

print(Fraction(1, 2) + Fraction(1, 3))
print(Fraction(2, 4) == Fraction(1, 2))
print(sorted([Fraction(3, 4), Fraction(1, 3), Fraction(5, 6), Fraction(1, 2)]))

5/6
True
[1/3, 1/2, 3/4, 5/6]


## Part D: Cumulative Project: Campus Bookshop

### Exercise 6: An Improved Book

In [10]:
class Book:
    TAX_RATE = 0.15

    def __init__(self, title, price, stock, category):
        self.title = title
        self.price = price
        self.stock = stock
        self.category = category

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = value

    @property
    def stock(self):
        return self._stock

    @stock.setter
    def stock(self, value):
        if value < 0:
            raise ValueError("stock cannot be negative")
        self._stock = value

    @classmethod
    def from_csv(cls, line):
        title, price, stock, category = line.strip().split(",")
        return cls(title, float(price), int(stock), category)

    def sell(self, quantity, member=False):
        if quantity > self.stock:
            print(f"Only {self.stock} copies of {self.title} available")
            return 0
        rate = (0.10 if quantity >= 10 else 0) + (0.05 if member else 0)
        self.stock -= quantity
        return round(self.price * quantity * (1 - rate) * (1 + Book.TAX_RATE), 2)

    def __str__(self):
        return f"{self.title}, {self.price:.2f} ({self.stock} in stock)"

    def __repr__(self):
        return f"Book({self.title!r}, {self.price}, {self.stock}, {self.category!r})"

    def __eq__(self, other):
        return self.title == other.title

    def __lt__(self, other):
        return self.price < other.price

lines = ["Python Basics,45.0,12,Programming", "Data Science Handbook,62.5,3,Data",
         "Statistics Made Simple,38.0,0,Data", "SQL in a Week,29.9,7,Programming"]
books = [Book.from_csv(l) for l in lines]
for b in sorted(books):
    print(b)
print(books[0].sell(12, member=True))
try:
    Book("Bad", -5, 1, "Data")
except ValueError as e:
    print("Refused:", e)

SQL in a Week, 29.90 (7 in stock)
Statistics Made Simple, 38.00 (0 in stock)
Python Basics, 45.00 (12 in stock)
Data Science Handbook, 62.50 (3 in stock)
527.85
Refused: price cannot be negative


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | `print` uses `__str__`, and falls back to `__repr__` if `__str__` is missing. |
| 2 | B | Sorting compares objects with `<`, which calls `__lt__`. |
| 3 | B | Class attributes are shared by all objects of the class. |
| 4 | B | A class method receives the class itself, conventionally called `cls`. |
| 5 | C | It is a convention telling other programmers not to use it directly. |